In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, DoubleType

# Bronze principle #1: we TELL Spark the schema instead of letting it guess
# (inferSchema). If tomorrow's file has a corrupted column, inference would
# silently adapt; an enforced schema FAILS LOUDLY. Loud > silent, remember?
schema = StructType([
    StructField("row_id", IntegerType()),
    StructField("SeriousDlqin2yrs", IntegerType()),
    StructField("RevolvingUtilizationOfUnsecuredLines", DoubleType()),
    StructField("age", IntegerType()),
    StructField("NumberOfTime30_59DaysPastDueNotWorse", IntegerType()),
    StructField("DebtRatio", DoubleType()),
    StructField("MonthlyIncome", DoubleType()),
    StructField("NumberOfOpenCreditLinesAndLoans", IntegerType()),
    StructField("NumberOfTimes90DaysLate", IntegerType()),
    StructField("NumberRealEstateLoansOrLines", IntegerType()),
    StructField("NumberOfTime60_89DaysPastDueNotWorse", IntegerType()),
    StructField("NumberOfDependents", IntegerType()),
])
# (note: hyphens in the original 30-59/60-89 names become underscores —
#  Delta column names can't contain '-'; a tiny, typical ingestion decision)

bronze_df = (spark.read
    .option("header", True)
    .schema(schema)
    .csv("/Volumes/workspace/default/raw/cs-training.csv"))

print(bronze_df.count(), "rows")
bronze_df.printSchema()

150000 rows
root
 |-- row_id: integer (nullable = true)
 |-- SeriousDlqin2yrs: integer (nullable = true)
 |-- RevolvingUtilizationOfUnsecuredLines: double (nullable = true)
 |-- age: integer (nullable = true)
 |-- NumberOfTime30_59DaysPastDueNotWorse: integer (nullable = true)
 |-- DebtRatio: double (nullable = true)
 |-- MonthlyIncome: double (nullable = true)
 |-- NumberOfOpenCreditLinesAndLoans: integer (nullable = true)
 |-- NumberOfTimes90DaysLate: integer (nullable = true)
 |-- NumberRealEstateLoansOrLines: integer (nullable = true)
 |-- NumberOfTime60_89DaysPastDueNotWorse: integer (nullable = true)
 |-- NumberOfDependents: integer (nullable = true)



In [0]:
from pyspark.sql import functions as F

# Bronze principle #2: land the data AS-IS (zeros ages, 96/98 codes and all)
# plus provenance columns. Bronze is the evidence locker; cleaning is silver's job.
bronze_df = (bronze_df
    .withColumn("_ingested_at", F.current_timestamp())
    .withColumn("_source_file", F.lit("cs-training.csv")))

bronze_df.write.mode("overwrite").saveAsTable("workspace.default.bronze_credit")
print("bronze_credit written")

bronze_credit written


In [0]:
%sql
DESCRIBE HISTORY workspace.default.bronze_credit

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
2,2026-07-22T11:41:42.000Z,73765391067652,sanjana.somashekar1999@gmail.com,CREATE OR REPLACE TABLE AS SELECT,"Map(isV1SaveAsTableOverwrite -> true, partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.format.version"":""2.12.0"",""delta.parquet.format.version.afe.internal"":""2.12.0"",""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true""}, statsOnLoad -> true)",null,List(2408342187648682),ff2aa6ea-2809-4b30-8803-ae72f2869472,0722-113540-qntp4xw2-v2n,1,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 1, numRemovedBytes -> 2618383, numDeletionVectorsRemoved -> 0, numOutputRows -> 150000, numOutputBytes -> 2618383)",null,Databricks-Runtime/18.x-aarch64-photon-scala2.13
1,2026-07-22T11:40:23.000Z,73765391067652,sanjana.somashekar1999@gmail.com,CREATE OR REPLACE TABLE AS SELECT,"Map(isV1SaveAsTableOverwrite -> true, partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.format.version"":""2.12.0"",""delta.parquet.format.version.afe.internal"":""2.12.0"",""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true""}, statsOnLoad -> true)",null,List(2408342187648682),14b8696f-5055-4ed3-b1cf-8b975af63ace,0722-113540-qntp4xw2-v2n,0,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 1, numRemovedBytes -> 2618383, numDeletionVectorsRemoved -> 0, numOutputRows -> 150000, numOutputBytes -> 2618383)",null,Databricks-Runtime/18.x-aarch64-photon-scala2.13
0,2026-07-22T11:37:54.000Z,73765391067652,sanjana.somashekar1999@gmail.com,CREATE OR REPLACE TABLE AS SELECT,"Map(isV1SaveAsTableOverwrite -> true, partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.format.version"":""2.12.0"",""delta.parquet.format.version.afe.internal"":""2.12.0"",""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true""}, statsOnLoad -> true)",null,List(2408342187648682),a11e98d0-9400-4a10-86a3-18d603e7da14,0722-113540-qntp4xw2-v2n,null,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 0, numRemovedBytes -> 0, numDeletionVectorsRemoved -> 0, numOutputRows -> 150000, numOutputBytes -> 2618383)",null,Databricks-Runtime/18.x-aarch64-photon-scala2.13


In [0]:
# DQ checks at the bronze->silver boundary. Bronze doesn't FIX, but it DETECTS
# and reports - this is the "monitoring checks data quality first" principle as code.
checks = {
    "row_count":        bronze_df.count(),
    "null_income":      bronze_df.filter(F.col("MonthlyIncome").isNull()).count(),
    "age_below_18":     bronze_df.filter(F.col("age") < 18).count(),
    "util_above_10":    bronze_df.filter(F.col("RevolvingUtilizationOfUnsecuredLines") > 10).count(),
    "special_codes_96_98": bronze_df.filter(
        F.col("NumberOfTime30_59DaysPastDueNotWorse").isin(96, 98) |
        F.col("NumberOfTimes90DaysLate").isin(96, 98) |
        F.col("NumberOfTime60_89DaysPastDueNotWorse").isin(96, 98)).count(),
}
for k, v in checks.items():
    print(f"{k:22s} {v}")

row_count              150000
null_income            29731
age_below_18           1
util_above_10          241
special_codes_96_98    269


In [0]:
%sql
SELECT COUNT(*) FROM workspace.default.bronze_credit VERSION AS OF 0

COUNT(*)
150000
